In [1]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re
import xml.etree.ElementTree as ET

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns, normalize_text
import xml.etree.ElementTree as ET

from pydantic_ai import Agent, RunContext
import asyncio
from pydantic import BaseModel, Field
import httpx
import requests
from enum import Enum

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_DIR = SILVER_DIR / "boe_candidates_docs_text"

BOE_CANDIDATES_DOCS_TEXT_PATH = BOE_CANDIDATES_DOCS_TEXT_DIR / "boe_candidates_docs_text.parquet"

BOE_AI_EXTRACTIONS_DIR = SILVER_DIR / "boe_ai_extractions"
BOE_AI_EXTRACTIONS_PATH = BOE_AI_EXTRACTIONS_DIR / "boe_ai_extractions.parquet"

GOLD_LIFECYCLE_EVENTS_DIR = GOLD_DIR / "lifecycle_events"
GOLD_LIFECYCLE_EVENTS_PATH = GOLD_LIFECYCLE_EVENTS_DIR / "lifecycle_events.parquet"

GOLD_ASSET_MENTIONS_DIR = GOLD_DIR / "asset_mentions"
GOLD_ASSET_MENTIONS_PATH = GOLD_ASSET_MENTIONS_DIR / "asset_mentions.parquet"

GOLD_ASSET_RELATION_MENTIONS_DIR = GOLD_DIR / "asset_relation_mentions"
GOLD_ASSET_RELATION_MENTIONS_PATH = GOLD_ASSET_RELATION_MENTIONS_DIR / "asset_relation_mentions.parquet"

GOLD_ADMINISTRATIVE_ACTIONS_DIR = GOLD_DIR / "administrative_actions"
GOLD_ADMINISTRATIVE_ACTIONS_PATH = GOLD_ADMINISTRATIVE_ACTIONS_DIR / "administrative_actions.parquet"

In [ ]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

df_test = df.loc[df["xml_status"] == "ok"]

df_test.head(2)

In [ ]:
# for _, row in df_test.sample(10, random_state=42).iterrows():
#     print("=" * 80)
#     print(row["identificador"])
#     print(row["titulo"])
#     print(row["texto_limpio"][:6000])

## Contratos de salida

Qué quiero saber de cada publicación?

### Relevancia energética

In [ ]:
class RelevanciaEnergetica(str, Enum):
    RELEVANTE = "relevante"
    NO_RELEVANTE = "no_relevante"
    DUDOSO = "dudoso"
    
# relevante:
#   Documento sobre generación eléctrica renovable, almacenamiento, evacuación, subestaciones, líneas eléctricas o autorizaciones ambientales/administrativas asociadas.

# no_relevante:
#   Documento energético genérico, normativo, estadístico, tarifario, presupuestario o no vinculado a un proyecto concreto.

# dudoso:
#   Documento con vocabulario energético, pero sin información suficiente para saber si corresponde a un proyecto tramitado.

### Tecnología

In [ ]:
class TechnologyType(str, Enum):
    FOTOVOLTAICA = "fotovoltaica"
    EOLICA = "eolica"
    TERMOSOLAR = "termosolar"
    HIDROELECTRICA = "hidroelectrica"
    GEOTERMICA = "geotermica"
    BIOMASA = "biomasa"
    BIOGAS = "biogas"
    HIDROGENO_VERDE = "hidrogeno_verde"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

class Technology(BaseModel):
    technology_type: TechnologyType
    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None
    description: str | None = None
    power_normalization_note: str | None = None

In [ ]:
class StorageSystem(BaseModel):
    exists: bool = False
    power_mw: float | None = None
    capacity_mwh: float | None = None

In [ ]:
class InfrastructureType(str, Enum):
    SUBESTACION = "subestacion"
    LINEA_ELECTRICA = "linea_electrica"
    CENTRO_SECCIONAMIENTO = "centro_seccionamiento"
    INFRAESTRUCTURA_EVACUACION = "infraestructura_evacuacion"
    OTRA = "otra"
    DESCONOCIDA = "desconocida"

class Infrastructure(BaseModel):
    type: InfrastructureType
    name: str | None = None
    voltage_kv: float | None = None
    length_km: float | None = None

In [ ]:
class HybridConfiguration(BaseModel):
    is_hybrid: bool = False
    generation_technology_types: list[TechnologyType] = Field(default_factory=list)
    includes_storage: bool = False
    description: str | None = None

### Localización

Usar Pydantic AI solo para extraer candidatos textuales y contexto; la validación final debe hacerla una función determinista.

In [ ]:
# Preparar tablas de referencia del INE

dim = pd.read_csv(BRONZE_DIR / "localizaciones_ine" / "codine_ccaaprovincia_20260614.csv", dtype=str)
dicc = pd.read_csv(BRONZE_DIR / "localizaciones_ine" / "diccionario26.csv", dtype=str)

# Normalizar ambas tablas añadiendo ceros
for col in ["codauto", "cpro"]:
    dim[col] = dim[col].str.zfill(2)
    dicc[col] = dicc[col].str.zfill(2)

municipios_ine_df = dicc.merge(
    dim[["codauto", "cpro", "comunidad_autonoma", "provincia"]],
    on=["codauto", "cpro"],
    how="left",
    validate="many_to_one",
)

municipios_ine_df = municipios_ine_df.rename(
    columns={"codauto": "cauto", "nombre": "municipio"}
)


# Normalizar
municipios_ine_df["municipio_norm"] = (
    municipios_ine_df["municipio"]
    .map(normalize_text)
)

municipios_ine_df["provincia_norm"] = (
    municipios_ine_df["provincia"]
    .map(normalize_text)
)

municipios_ine_df["comunidad_autonoma_norm"] = (
    municipios_ine_df["comunidad_autonoma"]
    .map(normalize_text)
)


# Organizar columnas
municipios_ine_df = municipios_ine_df[
    [
    "cauto", "comunidad_autonoma", "comunidad_autonoma_norm",
    "cpro", "provincia", "provincia_norm",
    "cmun", "municipio", "municipio_norm"
    ]
]

# Municipios duplicados
municipios_ine_df.loc[
    municipios_ine_df["municipio"].duplicated(keep=False)
].sort_values("municipio")


municipios_ine_df.head(3)


In [ ]:
# TODO: Afinar identificación de municipios.
# Requiere que municipios_ine_df esté cargado previamente y contenga:
# municipality, province, autonomous_community,
# ine_municipality_code, ine_province_code,
# municipality_norm, province_norm.

class MunicipalityResolutionStatus(str, Enum):
    RESOLVED = "resolved"
    AMBIGUOUS = "ambiguous"
    NOT_FOUND = "not_found"


class MunicipalityLocation(BaseModel):
    ine_municipality_code: str
    municipality: str
    ine_province_code: str
    province: str
    ine_autonomous_community_code: str | None = None
    autonomous_community: str
    
class MunicipalityLookupResult(BaseModel):
    query: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None

    resolution_status: MunicipalityResolutionStatus
    resolved: MunicipalityLocation | None = None
    candidates: list[MunicipalityLocation] = Field(default_factory=list)

    matched_by: str | None = None
    reason: str | None = None

### Procedimiento y estado

In [ ]:
class ProcedureStage(str, Enum):
    # Inicio y antecedentes del procedimiento
    SOLICITUD_TRAMITACION = "solicitud_tramitacion"
    SOLICITUD_TRAMITACION_AMBIENTAL = "solicitud_tramitacion_ambiental"
    SUBSANACION_DOCUMENTACION = "subsanacion_documentacion"
    VERIFICACION_REQUISITOS_TRAMITACION = "verificacion_requisitos_tramitacion"

    # Información pública
    INFORMACION_PUBLICA = "informacion_publica"

    # Evaluación ambiental
    DECLARACION_IMPACTO_AMBIENTAL = "declaracion_impacto_ambiental"
    INFORME_DETERMINACION_AFECCION_AMBIENTAL = "informe_determinacion_afeccion_ambiental"

    # Autorizaciones energéticas
    AUTORIZACION_ADMINISTRATIVA_PREVIA = "autorizacion_administrativa_previa"
    AUTORIZACION_ADMINISTRATIVA_CONSTRUCCION = "autorizacion_administrativa_construccion"
    AUTORIZACION_EXPLOTACION = "autorizacion_explotacion"

    # Utilidad pública y expropiación
    DECLARACION_UTILIDAD_PUBLICA = "declaracion_utilidad_publica"
    EXPROPIACION_FORZOSA = "expropiacion_forzosa"
    RELACION_BIENES_DERECHOS_AFECTADOS = "relacion_bienes_derechos_afectados"
    LEVANTAMIENTO_ACTAS_PREVIAS_OCUPACION = "levantamiento_actas_previas_ocupacion"
    ACTAS_OCUPACION = "actas_ocupacion"

    # Modificaciones y terminación anormal
    MODIFICACION = "modificacion"
    ARCHIVO_EXPEDIENTE = "archivo_expediente"
    DESISTIMIENTO = "desistimiento"
    INADMISION = "inadmision"

    # Fallback
    OTRO = "otro"
    NO_CONSTA = "no_consta"

In [ ]:
class ProcedureDecision(str, Enum):
    # Inicio y tramitación del expediente
    SOLICITADO = "solicitado"
    SUBSANADO = "subsanado"
    REQUISITOS_VERIFICADOS = "requisitos_verificados"
    MODIFICADO = "modificado"
    PRORROGADO = "prorrogado"
    FORMULADO = "formulado"

    # Información pública y participación
    SOMETIDO_INFORMACION_PUBLICA = "sometido_informacion_publica"
    CONVOCADO = "convocado"

    # Evaluación ambiental
    FAVORABLE = "favorable"
    DESFAVORABLE = "desfavorable"
    SOMETIDO_EIA_ORDINARIA = "sometido_eia_ordinaria"
    NO_SOMETIDO_EIA_ORDINARIA = "no_sometido_eia_ordinaria"
    DECLARADO_UTILIDAD_PUBLICA = "declarado_utilidad_publica"

    # Resolución administrativa
    APROBADO = "aprobado"
    AUTORIZADO = "autorizado"
    DENEGADO = "denegado"

    # Terminación anormal del procedimiento
    ARCHIVADO = "archivado"
    DESISTIDO = "desistido"
    INADMITIDO = "inadmitido"

    # Información no disponible
    NO_CONSTA = "no_consta"

In [ ]:
class ProcedureEvent(BaseModel):
    procedure_stage: ProcedureStage
    decision: ProcedureDecision = ProcedureDecision.NO_CONSTA
    evidence: str | None = None

### Proyecto

In [ ]:
class AssetRole(str, Enum):
    NEW_ASSET = "new_asset"
    EXISTING_ASSET = "existing_asset"
    MODIFIED_ASSET = "modified_asset"
    AFFECTED_ASSET = "affected_asset"
    ASSOCIATED_ASSET = "associated_asset"
    MAIN_ASSET = "main_asset"
    UNKNOWN = "unknown"


class AssetStatus(str, Enum):
    PLANNED = "planned"
    UNDER_PERMITTING = "under_permitting"
    AUTHORIZED = "authorized"
    UNDER_CONSTRUCTION = "under_construction"
    EXISTING = "existing"
    DENIED = "denied"
    ARCHIVED = "archived"
    UNKNOWN = "unknown"


class LifecycleEventType(str, Enum):
    NEW_PROJECT = "new_project"
    HYBRIDIZATION = "hybridization"
    MODIFICATION = "modification"
    EXPANSION = "expansion"
    REPOWERING = "repowering"
    STORAGE_ADDITION = "storage_addition"
    EVACUATION_INFRASTRUCTURE = "evacuation_infrastructure"
    OWNERSHIP_CHANGE = "ownership_change"
    OTHER = "other"
    UNKNOWN = "unknown"


class AssetRelationType(str, Enum):
    HYBRIDIZES_WITH = "hybridizes_with"
    ADDS_TECHNOLOGY_TO = "adds_technology_to"
    MODIFIES = "modifies"
    EXPANDS = "expands"
    REPOWERS = "repowers"
    ADDS_STORAGE_TO = "adds_storage_to"
    SHARES_GRID_ACCESS_WITH = "shares_grid_access_with"
    ASSOCIATED_WITH = "associated_with"
    SAME_PROJECT_GROUP_AS = "same_project_group_as"
    UNKNOWN = "unknown"


class ParticipantRole(str, Enum):
    PROMOTER = "promoter"
    CO_PROMOTER = "co_promoter"
    OWNER = "owner"
    OPERATOR = "operator"
    GRID_OWNER = "grid_owner"
    ADMINISTRATION = "administration"
    UNKNOWN = "unknown"


class ProjectParticipant(BaseModel):
    name: str
    role: ParticipantRole = ParticipantRole.UNKNOWN
    evidence: str | None = None


class EnergyAsset(BaseModel):
    local_asset_id: str
    name: str | None = None
    aliases: list[str] = Field(default_factory=list)

    role_in_event: AssetRole = AssetRole.UNKNOWN
    status_in_document: AssetStatus = AssetStatus.UNKNOWN

    technologies: list[Technology] = Field(default_factory=list)
    storage_systems: list[StorageSystem] = Field(default_factory=list)
    participants: list[ProjectParticipant] = Field(default_factory=list)
    locations: list[MunicipalityLocation] = Field(default_factory=list)

    evidence: str | None = None


class AssetRelation(BaseModel):
    source_asset_id: str
    target_asset_id: str
    relation_type: AssetRelationType
    evidence: str | None = None


class AdministrativeAction(BaseModel):
    stage: ProcedureStage
    decision: ProcedureDecision
    evidence: str | None = None


class ProjectLifecycleEvent(BaseModel):
    event_type: LifecycleEventType

    administrative_actions: list[AdministrativeAction]

    assets: list[EnergyAsset] = Field(default_factory=list)
    asset_relations: list[AssetRelation] = Field(default_factory=list)

    event_summary: str | None = None
    evidence: str | None = None

### Documento global

In [ ]:
class BOEProjectExtraction(BaseModel):
    identificador_boe: str
    fecha_publicacion: date | None = None

    relevancia_energetica: RelevanciaEnergetica
    es_relevante_para_proyecto: bool
    relevance_reason: str | None = None

    lifecycle_events: list[ProjectLifecycleEvent] = Field(default_factory=list)

    grouping_candidates: list[str] = Field(default_factory=list)
    grouping_notes: str | None = None

    extraction_notes: str | None = None

### Notas para la estimación objetiva de la confianza

In [ ]:
# TODO: Estimar objetivamente la confianza de la extracción con IA
# confidence = 1.0

# confidence = 1.0

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# if project_name is None:
#     confidence -= 0.2

# if promoter is None:
#     confidence -= 0.1

# if len(main_events) == 0:
#     confidence -= 0.3

# if len(locations) == 0:
#     confidence -= 0.1

# df[
#     (df["relevance_confidence"] < 0.7)
#     | (df["extraction_confidence"] < 0.7)
# ]

## Agente

In [ ]:
from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider

model = OllamaModel(
    "llama3.2:3b",
    #"qwen3:8b",
    provider=OllamaProvider(
        base_url="http://localhost:11434/v1"
    ),
)

agent = Agent(model)

result = await agent.run("¿Cuál es la capital de Italia? Responde en inglés.")

print(result.output)

La capital de Italia es Roma.


In [ ]:
agent = Agent(
    "google:gemini-2.5-flash",
    output_type=BOEProjectExtraction,
    instructions="""
Eres un extractor de información estructurada de documentos del BOE sobre proyectos energéticos.

Objetivo:
Extraer eventos de ciclo de vida de proyectos energéticos a partir de una publicación del BOE.

Reglas generales:
- Extrae únicamente información explícitamente contenida en el título o en el texto del documento.
- No inventes, completes ni corrijas datos por conocimiento externo.
- Si un dato no aparece, usa null, lista vacía o no_consta, según corresponda al esquema.
- Toda información relevante debe estar respaldada por evidencia textual.
- No asignes identificadores definitivos de proyecto.
- Usa solo local_asset_id internos al documento: asset_1, asset_2, asset_3.
- event_summary debe contener una frase breve, no nula, que resuma el evento principal.

Interpreta los números con formato español:
- "31,172 MW" equivale a 31.172 MW.
- "28.000 kW" equivale a 28000 kW.
- Cuando el texto incluya potencia unitaria y número de equipos, calcula la potencia total normalizada si la equivalencia es explícita.
- Si existe discrepancia aparente entre el cálculo explícito y una cifra textual, no afirmes que el BOE contiene una errata.
- Conserva la discrepancia en power_normalization_note.
- Ejemplo: "14 aerogeneradores de 2000 kW" se normaliza como 28 MW; si el texto además dice "28.000 MW", indícalo en la nota sin corregir ni calificar el texto.

Unidad de extracción:
- La unidad principal es lifecycle_events.
- Cada publicación BOE debe generar normalmente un único lifecycle_event principal.
- Un lifecycle_event representa el hito administrativo publicado en el BOE y su efecto sobre uno o varios activos energéticos.
- No crees un lifecycle_event separado para el trámite administrativo si ya está recogido en administrative_stage.
- Por ejemplo, si el documento formula un informe de determinación de afección ambiental para una hibridación, usa:
  event_type = hybridization
  administrative_stage = informe_determinacion_afeccion_ambiental
  administrative_decision = formulado
- No añadas otro lifecycle_event con event_type = environmental_assessment para el mismo hecho.

Activos:
Extrae solo activos relevantes para entender la evolución del proyecto:
- nuevas instalaciones de generación;
- instalaciones existentes afectadas por hibridación, modificación, ampliación o repotenciación;
- sistemas de almacenamiento;
- infraestructuras energéticas solo si son el objeto principal del documento.

No extraigas líneas, subestaciones, centros de seccionamiento ni puntos de conexión como assets si solo aparecen como infraestructura auxiliar de evacuación, conexión, modificación técnica o acceso a red.
Tampoco crees asset_relations hacia esas infraestructuras auxiliares.

Relaciones entre activos:
Usa asset_relations únicamente entre activos energéticos principales o activos existentes afectados por la evolución del proyecto.

Tipos de relación:
- hybridizes_with: una instalación nueva hibrida con una instalación existente.
- adds_technology_to: se añade una nueva tecnología a un activo o complejo existente.
- modifies: se modifica un activo existente.
- expands: se amplía un activo existente.
- repowers: se repotencia un activo existente.
- adds_storage_to: se añade almacenamiento a un activo existente.
- same_project_group_as: el texto indica que varios activos forman parte del mismo proyecto o complejo.
- associated_with: relación explícita relevante que no encaja en las anteriores.

No uses asset_relations para describir la red de evacuación, conexión o acceso.

Hibridación:
Si el documento describe una hibridación:
- crea un asset para la nueva instalación;
- crea otro asset para el activo existente afectado, si aparece en el texto;
- marca la nueva instalación como new_asset;
- marca el activo previo como existing_asset;
- crea una relación hybridizes_with desde la nueva instalación hacia el activo existente;
- no crees assets ni relaciones para subestaciones, líneas o puntos de conexión auxiliares;
- si el texto indica que la nueva tecnología se añade al complejo o activo existente, añade también adds_technology_to.

Promotores y participantes:
- Extrae promotores, copromotores, titulares u operadores solo si aparecen explícitamente.
- Puede haber varios participantes.
- No asumas que el promotor de un activo es también promotor de otro si el texto no lo dice.

Municipios:
Para cada municipio identificado:
1. Extrae el nombre del municipio mencionado en el documento.
2. Extrae provincia y comunidad autónoma solo si aparecen en el documento.
3. Llama siempre a la herramienta resolve_municipality.
4. Utiliza exclusivamente el resultado devuelto por resolve_municipality para completar municipio oficial, provincia, comunidad autónoma y códigos INE.
5. No inventes códigos INE ni divisiones administrativas.
6. No incluyas municipios que no aparezcan explícitamente como ubicación del activo.

Procedimiento administrativo:
- administrative_stage debe reflejar el trámite principal publicado en el BOE.
- administrative_decision debe reflejar la decisión principal: formulado, autorizado, denegado, sometido_informacion_publica, archivado, etc.
- No incluyas antecedentes menores salvo que sean necesarios para interpretar el evento.
- No dupliques como lifecycle_event lo que ya esté expresado como administrative_stage o administrative_decision.

Agrupación:
- grouping_candidates debe contener cadenas normalizadas útiles para agrupar publicaciones futuras.
- Incluye nombres de activos principales, nombres base compartidos, municipio/provincia y promotor si constan.
- No incluyas subestaciones, líneas ni puntos de conexión auxiliares como grouping_candidates salvo que sean el objeto principal del documento.
- No inventes un project_group_id definitivo.
"""
)

### Herramientas del agente

In [ ]:
# Palabras con poco valor discriminante para identificar municipios.
STOP_TOKENS = {
    "a", "de", "del", "el", "en", "la", "las", "los", "y",
    "municipio", "municipal", "termino",
}


def text_tokens(text: str | None) -> set[str]:
    """
    Convierte un texto en un conjunto de tokens normalizados,
    eliminando palabras poco informativas.
    """
    return {
        token
        for token in normalize_text(text).split()
        if token not in STOP_TOKENS
    }


def token_overlap_score(query: str | None, candidate: str | None) -> float:
    """
    Calcula la proporción de tokens de la consulta presentes
    en el candidato.

    Valor entre 0 y 1.
    """
    query_tokens = text_tokens(query)
    candidate_tokens = text_tokens(candidate)

    if not query_tokens or not candidate_tokens:
        return 0.0

    return len(query_tokens & candidate_tokens) / len(query_tokens)


def municipality_token_matches(
    municipality_name: str,
    candidate_municipality: str,
    *,
    allow_single_token: bool,
) -> bool:
    """
    Determina si un municipio candidato es compatible con la consulta.

    Reglas:
    - Coincidencia total de tokens -> match.
    - Coincidencia >= 80 % para consultas con varios tokens -> match.
    - Consultas de un solo token solo se aceptan si existen hints
      adicionales (provincia o comunidad autónoma).
    """
    query_tokens = text_tokens(municipality_name)
    candidate_tokens = text_tokens(candidate_municipality)

    if not query_tokens or not candidate_tokens:
        return False

    if query_tokens.issubset(candidate_tokens):
        return True

    if len(query_tokens) >= 2:
        return token_overlap_score(municipality_name, candidate_municipality) >= 0.8

    return allow_single_token and bool(query_tokens & candidate_tokens)


def hint_token_matches(
    hint: str | None,
    candidate: str | None,
) -> bool:
    """
    Comprueba si un hint administrativo (provincia o comunidad autónoma)
    comparte al menos un token relevante con el candidato.
    """
    if hint is None:
        return True

    hint_tokens = text_tokens(hint)
    candidate_tokens = text_tokens(candidate)

    if not hint_tokens or not candidate_tokens:
        return False

    return bool(hint_tokens & candidate_tokens)


def _row_to_location(row: pd.Series) -> MunicipalityLocation:
    """
    Convierte una fila de la dimensión INE en un objeto tipado.
    """
    return MunicipalityLocation(
        municipality=row["municipio"],
        province=row["provincia"],
        autonomous_community=row["comunidad_autonoma"],
        ine_municipality_code=row["cpro"] + row["cmun"],
        ine_province_code=row["cpro"],
        ine_autonomous_community_code=row["cauto"],
    )


def _build_lookup_result(
    municipality_name: str,
    province_hint: str | None,
    autonomous_community_hint: str | None,
    matches: pd.DataFrame,
    matched_by: str,
    reason: str,
) -> MunicipalityLookupResult:
    """
    Construye la respuesta final a partir de las coincidencias obtenidas.

    - 1 coincidencia  -> RESOLVED
    - >1 coincidencia -> AMBIGUOUS
    - 0 coincidencias -> NOT_FOUND
    """
    matches = matches.drop_duplicates(
        subset=["cauto", "cpro", "cmun"]
    )

    if len(matches) == 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.RESOLVED,
            resolved=_row_to_location(matches.iloc[0]),
            matched_by=matched_by,
            reason=reason,
        )

    if len(matches) > 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.AMBIGUOUS,
            candidates=[_row_to_location(row) for _, row in matches.iterrows()],
            matched_by=matched_by,
            reason=(
                "Existen varias coincidencias compatibles con los criterios "
                "proporcionados."
            ),
        )

    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia en el catálogo INE.",
    )


def resolve_municipality_impl(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    # Normalizar consulta y hints para matching.
    municipality_name_norm = normalize_text(municipality_name)
    province_hint_norm = normalize_text(province_hint) if province_hint else None
    autonomous_community_hint_norm = (
        normalize_text(autonomous_community_hint)
        if autonomous_community_hint
        else None
    )

    # Fase 1: coincidencia exacta por nombre normalizado de municipio.
    matches = municipios_ine_df.loc[
        municipios_ine_df["municipio_norm"] == municipality_name_norm
    ]

    # Fase 2: desambiguar coincidencias exactas mediante provincia.
    if not matches.empty and province_hint_norm:
        province_matches = matches.loc[
            matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_matches,
                matched_by="municipality_exact_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 3: desambiguar coincidencias exactas mediante comunidad autónoma.
    if not matches.empty and autonomous_community_hint_norm:
        ac_matches = matches.loc[
            matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_matches,
                matched_by="municipality_exact_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia exacta de municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 4: si la coincidencia exacta ya es única, resolver.
    if not matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=matches,
            matched_by="municipality_exact",
            reason="Municipio resuelto por coincidencia exacta.",
        )

    # Permitir búsquedas de un solo token únicamente cuando existen hints.
    allow_single_token = (
        province_hint is not None
        or autonomous_community_hint is not None
    )

    # Fase 5: búsqueda flexible por tokens del municipio.
    partial_matches = municipios_ine_df.loc[
        municipios_ine_df["municipio"].map(
            lambda value: municipality_token_matches(
                municipality_name,
                value,
                allow_single_token=allow_single_token,
            )
        )
    ]

    # Fase 6: filtrar coincidencias parciales mediante provincia.
    if not partial_matches.empty and province_hint is not None:
        province_partial_matches = partial_matches.loc[
            partial_matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_partial_matches,
                matched_by="municipality_token_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 7: filtrar coincidencias parciales mediante comunidad autónoma.
    if not partial_matches.empty and autonomous_community_hint is not None:
        ac_partial_matches = partial_matches.loc[
            partial_matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_partial_matches,
                matched_by="municipality_token_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 8: devolver coincidencias parciales restantes.
    if not partial_matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=partial_matches,
            matched_by="municipality_token",
            reason=(
                "Existen coincidencias por tokens del municipio, pero no hay "
                "hints suficientes para garantizar una resolución única."
            ),
        )

    # Fase 9: sin coincidencias exactas ni parciales.
    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia exacta ni por tokens en el catálogo INE.",
    )

In [ ]:
@agent.tool_plain
def resolve_municipality(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    return resolve_municipality_impl(
        municipality_name=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
    )

# resolve_municipality("Amurrio")
# resolve_municipality("Agurain/Salvatierra")
# resolve_municipality("Palmas")
# resolve_municipality("Palmas", province_hint="Las Palmas")
resolve_municipality("Gran Canaria", province_hint="Las Palmas")
# resolve_municipality("Palmas", autonomous_community_hint="Canarias")

## Prueba

In [ ]:
df_test.values[7]

In [ ]:
df_test2 = df_test.loc[df_test["identificador"] == "BOE-A-2023-10304"]

In [ ]:
for _, row in df_test2.sample(1, random_state=42).iterrows():
    prompt = f"""
Identificador BOE: {row["identificador"]}
Fecha publicación: {row["fecha_publicacion"]}
Título: {row["titulo"]}

Texto:
{row["texto_limpio"][:12000]}
"""
    result = await agent.run(prompt)

In [ ]:
resultoutput = result.output

my_result = result.output.model_dump_json()

my_result_dict = json.loads(my_result)

In [ ]:
print(
    json.dumps(
        my_result_dict,
        indent=2,
        ensure_ascii=False,
    )
)

In [ ]:
def build_ai_extraction_record(
    row: pd.Series,
    extraction: BOEProjectExtraction,
) -> dict:
    return {
        "identificador_boe": row["identificador"],
        "fecha_publicacion": row["fecha_publicacion"],
        "titulo": row["titulo"],
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": "google:gemini-2.5-flash",
        "extraction_status": "ok",
        "parse_error": None,
    }

In [ ]:
record = build_ai_extraction_record(row, result.output)

boe_ai_extractions = pd.DataFrame([record])

save_parquet(
    boe_ai_extractions,
    BOE_AI_EXTRACTIONS_PATH,
)

In [ ]:
pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)

In [ ]:
def flatten_lifecycle_events(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            records.append(
                {
                    "event_id": event_id,
                    "identificador_boe": extraction.identificador_boe,
                    "fecha_publicacion": extraction.fecha_publicacion,
                    "event_index": event_idx,
                    "event_type": event.event_type.value,
                    "event_summary": event.event_summary,
                    "evidence": event.evidence,
                }
            )

    return pd.DataFrame(records)

In [ ]:
def flatten_administrative_actions(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )

        for event_idx, event in enumerate(
            extraction.lifecycle_events,
            start=1,
        ):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for action_idx, action in enumerate(
                event.administrative_actions,
                start=1,
            ):
                action_id = (
                    f"{extraction.identificador_boe}"
                    f"_event_{event_idx}"
                    f"_action_{action_idx}"
                )

                records.append(
                    {
                        "action_id": action_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.identificador_boe,
                        "fecha_publicacion": extraction.fecha_publicacion,
                        "action_index": action_idx,
                        "administrative_stage": action.stage.value,
                        "administrative_decision": action.decision.value,
                        "evidence": action.evidence,
                    }
                )

    return pd.DataFrame(records)

In [ ]:
ai_extractions = pd.read_parquet(BOE_AI_EXTRACTIONS_PATH)

lifecycle_events = flatten_lifecycle_events(ai_extractions)
lifecycle_events

In [ ]:
administrative_actions = flatten_administrative_actions(ai_extractions)
administrative_actions

In [ ]:
save_parquet(lifecycle_events, GOLD_LIFECYCLE_EVENTS_PATH)
save_parquet(administrative_actions, GOLD_ADMINISTRATIVE_ACTIONS_PATH)

In [ ]:
ai_extractions = pd.read_parquet(GOLD_LIFECYCLE_EVENTS_PATH)
display(ai_extractions)

In [ ]:
# TODO: Almacenar resultados en .parquet
# TODO: Comparar resultados con lo obtenido de la API directamente (contrastar)